### Test des tables possibles de pipeline param (test avec class simple)

In [324]:
from enum import Enum
from typing import Optional
from datetime import datetime

class TypeDonnees(Enum):
    INT = "int"
    DECIMAL = "decimal"
    STRING = "string"
    DATE = "date"
    TIMESTAMP = "timestamp"
    BOOLEAN = "boolean"

# Classe de base pour les contraintes
class ColumnConstraint:
    def __init__(self, id_constraint: int):
        self.id_constraint = id_constraint

# Contraintes STRING
class StringConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, min_length: int = 0, max_length: int = None):
        super().__init__(id_constraint)
        self.min_length = min_length
        self.max_length = max_length

# Contraintes NUMERIC
class NumericConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int, nb_min: float = None,
                 nb_max: float = None, nb_decimal: int = None):
        super().__init__(id_constraint)
        self.nb_min = nb_min
        self.nb_max = nb_max
        self.nb_decimal = nb_decimal

# Contraintes DATE
class DateConstraint(ColumnConstraint):
    def __init__(self, id_constraint: int,
                 date_min: datetime = None,
                 date_max: datetime = None):
        super().__init__(id_constraint)
        self.date_min = date_min
        self.date_max = date_max

# ETLColumnMapping (classe principale)
# Ici on compose avec une contrainte (et pas héritage direct)
class ETLColumnMapping:
    def __init__(self,
                 id_etl_column_mapping: int,
                 colonne_bdd: str,
                 colonne_fichier: str,
                 in_file: bool,
                 type_donnees: TypeDonnees,
                 nullable: bool,
                 valeur_defaut: str,
                 unique_constraint: bool,
                 constraint: ColumnConstraint = None):
        
        self.id_etl_column_mapping = id_etl_column_mapping
        self.colonne_bdd = colonne_bdd
        self.colonne_fichier = colonne_fichier
        self.in_file = in_file
        self.type_donnees = type_donnees
        self.nullable = nullable
        self.valeur_defaut = valeur_defaut
        self.unique_constraint = unique_constraint
        
        # la contrainte dépend du type
        self.constraint = constraint



class ExtensionFichier(Enum):
    CSV = "csv"
    JSON = "json"

class PipelineETL:
    def __init__(self, id_etl_pipeline: int, libelle: str, table_nom: str,
                 dossier_emplacement: str, nom_fichier_fixe: str,
                 nom_fichier_variable: str,
                 extension_fichier: ExtensionFichier,
                 active: bool,
                 colonnes: list[ETLColumnMapping] = None):

        self.id_etl_pipeline = id_etl_pipeline
        self.libelle = libelle
        self.table_nom = table_nom
        self.dossier_emplacement = dossier_emplacement
        self.nom_fichier_fixe = nom_fichier_fixe
        self.nom_fichier_variable = nom_fichier_variable
        self.extension_fichier = extension_fichier
        self.active = active
        self.colonnes = colonnes or []

# MOOK BDD
def get_pipelineBDD_TODO():
    col1 = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="nom",
        colonne_fichier="name",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut=None,
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=2, max_length=50)
    )
    col2 = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="equipement",
        colonne_fichier="equipments",
        in_file=True,
        type_donnees=TypeDonnees.STRING,
        nullable=False,
        valeur_defaut="test valeur par defaut",
        unique_constraint=False,
        constraint=StringConstraint(1, min_length=2, max_length=255)
    )
    col3 = ETLColumnMapping(
        id_etl_column_mapping=1,
        colonne_bdd="proteines",
        colonne_fichier="Calories (kcal)",
        in_file=True,
        type_donnees=TypeDonnees.DECIMAL,
        nullable=False,
        valeur_defaut=900,
        unique_constraint=False,
        constraint=NumericConstraint(1, nb_min=-999.9, nb_max=999.9, nb_decimal=1)
    )
    pipeline1 = PipelineETL(
        1, 
        "Import Aliment 2025Q1", 
        "aliment", 
        "\\raw\\", 
        "exercisedb_hobby", 
        "", 
        ExtensionFichier.JSON, 
        True,
        [col1, col2, col3]
    )
    return pipeline1

# Le but de ce notebook

L’objectif ici est de créer des fonctions réutilisables qui seront exploitées dans des pipelines de traitement de données.

Ces fonctions couvrent plusieurs étapes :

- Ouverture et lecture de fichiers (CSV / JSON)
- Nettoyage et vérification des données
- Séparation des lignes valides des anomalies

Ces fonctions doivent être le plus génériques possible, sans dépendre de valeurs codées en dur, afin d’être facilement réutilisables.

J’expérimente l’idée d’un pipeline "écrit en BDD", en schématisant les données plutôt qu’en utilisant de réelles bases de données, pour tester le concept.

## Partie 1 : Ouverture de fichiers

Les fonctions de cette partie servent à ouvrir un ou plusieurs fichiers CSV ou JSON dans le cadre d’une pipeline. Elles permettent :

- De localiser les fichiers dans les dossiers
- De détecter leur type (CSV / JSON)
- De les convertir en DataFrame avec pandas, pour les exploiter ensuite sans se soucier des extensions

In [325]:
import os
import re
import pandas as pd
import numpy as np

DATA_ROOT = "../data"

### 1.1 Normalisation du chemin

Cette fonction permet de déterminer le dossier où se trouvent les fichiers à traiter.

- Elle se base sur une variable d’environnement comme DATA_ROOT
- On peut ensuite fournir en argument des sous-dossiers où l’on imagine stocker les fichiers CSV ou JSON

Cela garantit que les pipelines restent flexibles et portables.

In [326]:
# 1. Normalisation du chemin
def normalize_path(dossier_emplacement, data_root=DATA_ROOT):
    """
    Normalise le chemin du dossier relatif à ./data/
    """
    # Normaliser slashes
    dossier_emplacement = dossier_emplacement.replace("\\", "/").replace("//", "/")
    # Supprimer slash en début ou fin
    dossier_emplacement = dossier_emplacement.strip("/")
    # Construire chemin complet
    full_path = os.path.join(data_root, dossier_emplacement)
    # Normaliser le chemin final
    return os.path.normpath(full_path)

### 1.2 Création du motif Regex pour trouver le fichier

Cette fonction génère un motif regex pour identifier les fichiers à traiter.

- Arguments : nom_fixe, nom_variable, extension
    - nom_fixe : première partie du nom du fichier (ex. "Exercice")
    - nom_variable : partie variable du nom du fichier (ex. "YYYYMMDD")
    - extension : type de fichier (csv ou json)

Le motif généré permet de retrouver tous les fichiers correspondants, par exemple :

- CSV commençant par "Exercice"
- Avec 8 caractères pour la date
- Se terminant par .csv

In [327]:
# 2. Création du motif regex pour le fichier
def build_filename_pattern(nom_fixe, nom_variable, extension):
    if not extension.startswith("."):
        extension = "." + extension
    
    if nom_variable:
        # Le nom variable a une longueur connue : on remplace par autant de "."
        motif = f"^{re.escape(nom_fixe)}{'.' * len(nom_variable)}{re.escape(extension)}$"
    else:
        motif = f"^{re.escape(nom_fixe)}{re.escape(extension)}$"
    
    return motif

### 1.3 Recherche des fichiers correspondants

Cette fonction cherche les fichiers dans un dossier en fonction de la regex fournie :

- Arguments : dossier et motif regex
- Retour : liste des chemins de fichiers correspondants

⚠️ Actuellement, un print est utilisé si le dossier n’existe pas.

- En production, il serait préférable de remonter une erreur ou de logger l’information.

In [328]:
# 3. Recherche des fichiers correspondants et retourne les path
def find_matching_files(dossier, motif_regex):
    matched_files = []
    if os.path.exists(dossier) and os.path.isdir(dossier):
        for f in os.listdir(dossier):
            if re.match(motif_regex, f):
                matched_files.append(os.path.join(dossier, f))  # chemin complet
    else:
        print(f"[WARNING] Le dossier '{dossier}' n'existe pas.")
    return matched_files


# --- TESTS ---
dossier_emplacement = "\\raw\\"
nom_fichier_fixe = "exercisedb_hobby"
nom_fichier_variable = ""  # par exemple une date
extension_fichier = ".json"

# Normalisation du chemin
normalized_folder = normalize_path(dossier_emplacement)
print("Chemin normalisé :", normalized_folder)

# Construction du motif regex
pattern = build_filename_pattern(nom_fichier_fixe, nom_fichier_variable, extension_fichier)
print("Motif regex :", pattern)

# Recherche des fichiers
matched = find_matching_files(normalized_folder, pattern)
print("Fichiers trouvés :", matched)


# --- Autre exemple ---
dossier_emplacement2 = "raw/"
nom_fichier_fixe2 = "exercisedb_hobby"
nom_fichier_variable2 = ""
extension_fichier2 = "json"

normalized_folder2 = normalize_path(dossier_emplacement2)
pattern2 = build_filename_pattern(nom_fichier_fixe2, nom_fichier_variable2, extension_fichier2)
matched2 = find_matching_files(normalized_folder2, pattern2)

print("\nChemin normalisé 2 :", normalized_folder2)
print("Motif regex 2 :", pattern2)
print("Fichiers trouvés 2 :", matched2)

Chemin normalisé : ..\data\raw
Motif regex : ^exercisedb_hobby\.json$
Fichiers trouvés : ['..\\data\\raw\\exercisedb_hobby.json']

Chemin normalisé 2 : ..\data\raw
Motif regex 2 : ^exercisedb_hobby\.json$
Fichiers trouvés 2 : ['..\\data\\raw\\exercisedb_hobby.json']


### 1.4 Ouverture et lecture des fichiers

Cette fonction prend un **chemin de fichier unique** et :

- Tente de le lire avec pandas selon son type (CSV ou JSON)
- Retourne un **DataFrame unique**
- Pour les CSV :
  - On considère que chaque fichier a une ligne header
  - Les lignes qui ne correspondent pas au nombre de colonnes du header sont automatiquement ignorées

⚠️ Actuellement, un print s’affiche si :

- Ce n’est ni un CSV ni un JSON
- Le fichier n’existe pas ou ne peut pas être lu
- En production, ces prints devraient être remplacés par des logs ou exceptions

💡 Notes complémentaires :

- Une seconde fonction `read_files` permet de traiter une **liste de fichiers**, mais elle utilise simplement `read_single_file` en boucle. Elle est moins utile si l’on souhaite suivre individuellement chaque fichier (logs, comptage des lignes ignorées, etc.)
- On pourrait également comptabiliser le nombre de lignes supprimées dans les CSV pour un suivi et un logging plus précis

In [329]:
# Lecture automatique avec pandas
def read_single_file_with_pandas(file_path):
    """
    Lit un fichier CSV ou JSON et retourne un DataFrame.
    Retourne None si l'extension n'est pas supportée ou en cas d'erreur.
    """
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".csv":
            df = pd.read_csv(file_path, header=0, on_bad_lines='skip') # header obligatoire et saute les lignes incorrectes (nb colonne different du header)
        elif ext == ".json":
            df = pd.read_json(file_path)
        else:
            print(f"[WARNING] Extension non supportée pour {file_path}, ignoré.")
            return None
        return df
    except Exception as e:
        print(f"[ERROR] Impossible de lire {file_path}: {e}")
        return None


def read_files_with_pandas(file_paths):
    """
    Lit une liste de fichiers CSV ou JSON et retourne une liste de DataFrames
    en utilisant read_single_file_with_pandas.
    """
    dfs = []
    for fpath in file_paths:
        df = read_single_file_with_pandas(fpath)
        if df is not None:
            dfs.append(df)
    return dfs


# ---TEST---
dfs = read_files_with_pandas(matched)
print(f"Nombre de DataFrames lus : {len(dfs)}")
if dfs:
    print(dfs[0].head())  # afficher les 5 premières lignes du premier fichier

Nombre de DataFrames lus : 1
             exerciseId                                   name  \
0  exr_41n2ha5iPFpN3hEJ                                          
1  exr_41n2haAabPyN5t8y                             Side Lunge   
2  exr_41n2hadPLLFRGvFk        Sliding Floor Pulldown on Towel   
3  exr_41n2hadQgEEX8wDN                            Triceps Dip   
4  exr_41n2haNJ3NA8yCE2  Dumbbell Incline One Arm Hammer Press   

                                            imageUrl              bodyParts  \
0  https://cdn.exercisedb.dev/media/w/images/EsVO...                [WAIST]   
1  https://cdn.exercisedb.dev/media/w/images/RB7N...   [QUADRICEPS, THIGHS]   
2  https://cdn.exercisedb.dev/media/w/images/9fW9...                 [BACK]   
3  https://cdn.exercisedb.dev/media/w/images/MruS...  [TRICEPS, UPPER ARMS]   
4  https://cdn.exercisedb.dev/media/w/images/qkXo...           [UPPER ARMS]   

      equipments exerciseType                  targetMuscles  \
0  [BODY WEIGHT]     STRENGTH      

### 1.5 Ptt équivalent au downloader ?

In [330]:
def get_df_matched_files(pipeline: PipelineETL):
    """
    À partir d'une instance PipelineETL, retourne la liste des fichiers/dataframe correspondants.
    """
    # 1. Normalisation du chemin
    normalized_folder = normalize_path(pipeline.dossier_emplacement)
    # print("[debug] get_df_matched_files - normalized_folder : " + normalized_folder)

    # 2. Construction du motif regex
    pattern = build_filename_pattern(
        pipeline.nom_fichier_fixe,
        pipeline.nom_fichier_variable,
        pipeline.extension_fichier.value  # Enum → string
    )
    # print("[debug] get_df_matched_files - pattern : " + pattern)

    # 3. Recherche des fichiers
    matched_files_path = find_matching_files(normalized_folder, pattern)
    # print("[debug] get_df_matched_files - Fichiers trouvés :", matched_files_path)

    # 4. Ouvre les fichier en retournant une liste de dataframe
    df_matched = read_files_with_pandas(matched_files_path)

    return df_matched

## Partie 2 : Nettoyage et vérification des données

### 2.1 ColumnMapper 

Responsabilité :

- renommer colonnes
- créer colonnes manquantes

Avec en argument un dataframe et une liste de la class [ETLColumnMapping]

In [331]:
def column_mapper(
    df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> pd.DataFrame:
    result = pd.DataFrame()

    for m in mappings:
        if m.in_file and m.colonne_fichier in df.columns:
            result[m.colonne_bdd] = df[m.colonne_fichier]
        else:
            result[m.colonne_bdd] = None

    return result

### 2.2 Créer un dataframe pour mettre les anomalies

In [332]:
def generate_anomaly_dataframe(mappings: list[ETLColumnMapping]) -> pd.DataFrame:
    columns = [m.colonne_bdd for m in mappings]
    columns += ["erreur"]
    
    result = pd.DataFrame(columns=columns)
    
    return result

### 2.3 Nettoie les colonnes texte d'un DataFrame (supprime les espaces inutiles)

In [333]:
def clean_txt(df):
    """
    Nettoie les colonnes texte d'un DataFrame :
    - Supprime les espaces en début et fin de chaque cellule.
    - Remplace les cellules vides ou ne contenant que des espaces par NaN.
    """
    # On travaille sur une copie pour ne pas modifier l'original
    df_clean = df.copy()
    
    # Sélectionner les colonnes texte
    text_columns = df_clean.select_dtypes(include='object').columns
    
    # Supprimer les espaces en début/fin
    df_clean[text_columns] = df_clean[text_columns].apply(lambda x: x.str.strip())
    
    # Remplacer les cellules vides ou contenant uniquement des espaces par NaN
    df_clean[text_columns] = df_clean[text_columns].replace(r'^\s*$', np.nan, regex=True)
    
    return df_clean

### 2.4 Verifier les valeurs null

In [334]:
def handle_missing_values(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Traite les valeurs manquantes dans df selon les règles de mappings :
    - Convertit les valeurs vides / null / NaN en pd.NA
    - Si nullable=True : laisse pd.NA
    - Si nullable=False et valeur_defaut définie : remplace les NA par la valeur par défaut
    - Sinon : les lignes avec NA deviennent anomalies
    """
    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        # --- Étape 1 : standardiser les valeurs "vides" en pd.NA ---
        series = df_clean[col].replace(["", " ", None, "null", "NULL"], pd.NA)

        # --- Étape 2 : vérifier nullable ---
        mask_na = series.isna()
        if not mask_na.any():
            # pas de valeur manquante → passer à la colonne suivante
            continue

        if mapping.nullable:
            # nullable = True → laisser pd.NA
            df_clean[col] = series
            continue

        if mapping.valeur_defaut not in [None, "", np.nan]:
            # nullable = False mais valeur par défaut définie → remplacer les NA
            df_clean[col] = series.fillna(mapping.valeur_defaut)
        else:
            # nullable = False et pas de valeur par défaut → ce sont des anomalies
            rows_with_error = df_clean.loc[mask_na].copy()
            rows_with_error["erreur"] = f"La cellule {col} est null"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
            # Supprimer ces lignes du df_clean
            df_clean = df_clean.loc[~mask_na]

        df_clean.reset_index(drop=True, inplace=True)
        anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

### 2.5 Regarder si les colonnes sont convertible dans le type cible

In [335]:
def convert_column_type(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    mappings: list[ETLColumnMapping]
) -> tuple[pd.DataFrame, pd.DataFrame]:

    df_clean = df.copy()
    anomalies = anomaly_df.copy()

    for mapping in mappings:
        col = mapping.colonne_bdd
        if col not in df_clean.columns:
            continue

        series = df_clean[col]

        #region --- Étape 1 : conversion selon type de toutes les lignes de la colonne ---
        if mapping.type_donnees == TypeDonnees.INT:
            converted = pd.to_numeric(series, errors='coerce').astype('Int64')
        elif mapping.type_donnees == TypeDonnees.DECIMAL:
            converted = pd.to_numeric(series, errors='coerce')
        elif mapping.type_donnees == TypeDonnees.STRING:
            converted = series.astype(str)
        elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
            converted = pd.to_datetime(series, errors='coerce')
            if mapping.type_donnees == TypeDonnees.DATE:
                converted = converted.dt.date
        elif mapping.type_donnees == TypeDonnees.BOOLEAN:
            converted = series.map(lambda x: True if str(x).lower() in ['true','1']
                                   else False if str(x).lower() in ['false','0'] else pd.NA)
        else:
            converted = series
        #endregion

        #region --- Étape 2 : gérer les valeurs manquantes selon nullable / valeur par défaut ---
        mask_invalid = converted.isna()

        # Si valeur null ok alors ne rien faire de plus
        if mapping.nullable:
            # On laisse les NaN
            df_clean[col] = converted
            continue

        if mapping.valeur_defaut not in [None, "", np.nan]:
            # Convertir valeur par défaut dans le bon type
            try:
                if mapping.type_donnees == TypeDonnees.INT:
                    default_value = pd.to_numeric(mapping.valeur_defaut)
                elif mapping.type_donnees == TypeDonnees.DECIMAL:
                    default_value = pd.to_numeric(mapping.valeur_defaut)
                elif mapping.type_donnees == TypeDonnees.STRING:
                    default_value = str(mapping.valeur_defaut)
                elif mapping.type_donnees in [TypeDonnees.DATE, TypeDonnees.TIMESTAMP]:
                    default_value = pd.to_datetime(mapping.valeur_defaut)
                    if mapping.type_donnees == TypeDonnees.DATE:
                        default_value = default_value.date()
                elif mapping.type_donnees == TypeDonnees.BOOLEAN:
                    default_value = True if str(mapping.valeur_defaut).lower() in ['true','1'] else False
                else:
                    default_value = mapping.valeur_defaut
            except Exception:
                # Valeur par défaut invalide : toutes les lignes invalides deviennent anomalies
                rows_with_error = df_clean.loc[mask_invalid].copy()
                rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible et la valeur par défaut '{mapping.valeur_defaut}' est invalide"
                anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
                # On supprime ces lignes
                df_clean = df_clean.loc[~mask_invalid]
                continue

            # Remplacer les NaN par valeur par défaut
            converted[mask_invalid] = default_value
            df_clean[col] = converted
        else:
            # Ni nullable, ni valeur par défaut : ce sont des anomalies
            rows_with_error = df_clean.loc[mask_invalid].copy()
            rows_with_error["erreur"] = f"La cellule {col} n'est pas convertible en {mapping.type_donnees.value}"
            anomalies = pd.concat([anomalies, rows_with_error], ignore_index=True)
            # Supprimer ces lignes du df_clean
            df_clean = df_clean.loc[~mask_invalid]
        #endregion

    df_clean.reset_index(drop=True, inplace=True)
    anomalies.reset_index(drop=True, inplace=True)

    return df_clean, anomalies

### Test

In [336]:
pipeline_rules = get_pipelineBDD_TODO()
pipeline_column_mapping = pipeline_rules.colonnes

dfs_matched_files = get_df_matched_files(pipeline_rules)

for df in dfs_matched_files:
    print("------------- Traitement d'un df ----------------------")
    df_clean = column_mapper(df, pipeline_column_mapping)
    anomalies = generate_anomaly_dataframe(pipeline_column_mapping)

    # supprime les espaces inutiles
    df_clean = clean_txt(df_clean)

    # Verifier les valeurs null 
    df_clean, anomalies = handle_missing_values(df_clean, anomalies, pipeline_column_mapping)

    # Regarder si les colonnes sont convertible dans le type cible
    df_clean, anomalies = convert_column_type(df_clean, anomalies, pipeline_column_mapping)

    print("----- df_clean -----")
    print(df_clean.head)
    print("----- anomalies -----")
    print(anomalies.head)

------------- Traitement d'un df ----------------------
----- df_clean -----
<bound method NDFrame.head of                                              nom              equipement  \
0                                     Side Lunge  test valeur par defaut   
1                Sliding Floor Pulldown on Towel  test valeur par defaut   
2                                    Triceps Dip  test valeur par defaut   
3          Dumbbell Incline One Arm Hammer Press  test valeur par defaut   
4                        Feet and Ankles Stretch  test valeur par defaut   
5                                     Jump Squat  test valeur par defaut   
6                 Dumbbell Single Leg Calf Raise  test valeur par defaut   
7                              Peroneals Stretch  test valeur par defaut   
8       Lying Double Legs Biceps Curl with Towel  test valeur par defaut   
9                          Seated Row with Towel  test valeur par defaut   
10  Calf Raise from Deficit with Chair Supported  test va

C:\Users\33631\AppData\Local\Temp\ipykernel_17568\925368360.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df_clean.select_dtypes(include='object').columns
